In [53]:
# installing libraries and packages
import os
import numpy as np
from PIL import Image, ImageFile

In [54]:
# global variables
data_directory = os.path.expanduser("~/.cache/kagglehub/datasets/abdelghaniaaba/wildfire-prediction-dataset/versions/1")
image_dimension = (64, 64)
classification = ["wildfire", "nowildfire"]

In [55]:
# Enables python imaging library to load truncated images
ImageFile.LOAD_TRUNCATED_IMAGES = True

# image processing tool
def load_images(split_dir, image_dimension, classification):
    # collects pixel arrays and a corresponding class label
    X, y = [], []
    
    # excavating class directories to assign class labels 
    for label, cls in enumerate(classification):
        cls_path = os.path.join(split_dir, cls)
        for filename in os.listdir(cls_path):
            filepath = os.path.join(cls_path, filename)
            # converting image data into acceptable format for processing
            img = Image.open(filepath).convert("RGB")
            img = img.resize(image_dimension)
            
            # scaling pixel values
            arr = np.array(img, dtype=np.float32) / 255.0
            
            # flatten to accommodate simple nature of MLP 
            X.append(arr.flatten())
            
            # adding label
            y.append(label)
    return np.array(X), np.array(y)

In [56]:
# creating directory splits for train, test, and valid samples 
X_train, y_train = load_images(os.path.join(data_directory, "train"), image_dimension, classification)
X_val,   y_val   = load_images(os.path.join(data_directory, "valid"), image_dimension, classification)
X_test,  y_test  = load_images(os.path.join(data_directory, "test"),  image_dimension, classification)

In [57]:
# Checking array dimensions
print("Checking Shapes-")
print(f"Train: {X_train.shape}")
print(f"Val: {X_val.shape}")
print(f"Test: {X_test.shape}")

Checking Shapes-
Train: (30250, 12288)
Val: (6300, 12288)
Test: (6300, 12288)


In [58]:
# Normalizing pixel data as MLP is sensitive to feature scale
mean = X_train.mean(axis=0)

# preventing division by zero
std  = X_train.std(axis=0) + 1e-8

# Z-score calculation
X_train = (X_train - mean) / std
X_val   = (X_val - mean) / std
X_test  = (X_test - mean) / std

In [18]:
'''
WORK IN PROGRESS BELOW...Past work template
'''
# adding additional libraries
import matplotlib.pyplot as plt
np.random.seed(8)

# activation functions
# https://www.geeksforgeeks.org/machine-learning/activation-functions-neural-networks/
def relu(z):
    return np.maximum(0, z)

# clipping to introduce stability when the model gets trained
def sigmoid(z):
    z = np.clip(z, -50, 50)
    return 1.0 / (1.0 + np.exp(-z))

In [19]:
# step size
# accommodates the higher dimensionality of image data
eta = 0.01

# max iterations
epochs = 500

# hidden layer nodes
h = 64

n_train, d = X_train.shape

# initializing weights
W1 = 0.1 * np.random.randn(d, h)
w2 = 0.1 * np.random.randn(h, 1)

# objective function
def f(x):
    # hidden layer
    z1 = W1.T @ x
    h = relu(z1)

    # output layer
    z2 = (w2.T @ h)[0, 0]
    return sigmoid(z2)

# binary classification (Front=0, Rear=1)
# binary cross-entropy (log loss); clip p to avoid log(0)
def log_loss(y_true, p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return -np.mean(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))

# performing batch gradient descent (accumulate gradients over all samples each epoch)
errors = []

for epoch in range(epochs):
    # accumulate gradients over the samples
    d_w2 = np.zeros_like(w2)
    d_W1 = np.zeros_like(W1)

    # loop through training data
    for i in range(n_train):
        x = X_train[i].reshape(d, 1)
        yi = y_train[i]

        # forward pass
        z1 = W1.T @ x
        h = relu(z1)

        # apply sigmoid sigmoid function for binary classifier
        pi = sigmoid((w2.T @ h)[0, 0])

        # error term
        error = (pi - yi)

        # gradient for w2
        d_w2 += (1/n_train) * error * h

        # ReLU application
        mask = (z1 > 0).astype(float)

        # gradient for W1
        d_W1 += (1/n_train) * error * (x @ (w2 * mask).T)

    # updating weights
    w2 = w2 - eta * d_w2
    W1 = W1 - eta * d_W1

    # compute training log loss 
    p_train = np.array([f(X_train[i].reshape(d, 1)) for i in range(n_train)])
    e = log_loss(y_train, p_train)
    errors.append(e)

print("W^(1):", W1)
print("w^(2):", w2)

# performing test set prediction
n_test = X_test.shape[0]
p_test = np.array([f(X_test[i].reshape(d, 1)) for i in range(n_test)])
yhat = (p_test >= 0.5).astype(int)

accuracy = np.mean(yhat == y_test)
print("Test accuracy:", accuracy)

# plotting loss / error over gradient descent
plt.plot(range(epochs), errors)
plt.title("Training log loss over GD")
plt.xlabel("epochs")
plt.ylabel("log loss")
plt.show()

KeyboardInterrupt: 

In [ ]:
# personal sanity check
np.mean(y_train), np.mean(y_test)

In [ ]:
W1

In [ ]:
w2